In [3]:
import os
os.chdir("G:\\PycharmProjects\\names-demo")
os.getcwd()

'G:\\PycharmProjects\\names-demo'

In [2]:
import pandas as pd
import os
import polars as pl
# folder containing your files
folder_path = "data/raw/names-usa/namesbystate"

# list all files (adjust extension if needed)
files = [f for f in os.listdir(folder_path) if f.endswith(".TXT")]

df_list = []

for file in files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path,header=None,names=["state", "gender", "year", "name", "count"])
    df_list.append(df)

# combine all files into one dataframe
df = pd.concat(df_list, ignore_index=True)
df.head()

,state,gender,year,name,count
0,AK,F,1910,Mary,14
1,AK,F,1910,Annie,12
2,AK,F,1910,Anna,10
3,AK,F,1910,Margaret,8
4,AK,F,1910,Helen,7


In [3]:
state_map = {
    'AK': 'Alaska', 'AL': 'Alabama', 'AR': 'Arkansas', 'AZ': 'Arizona',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut',
    'DC': 'District of Columbia', 'DE': 'Delaware', 'FL': 'Florida',
    'GA': 'Georgia', 'HI': 'Hawaii', 'IA': 'Iowa', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'KS': 'Kansas', 'KY': 'Kentucky',
    'LA': 'Louisiana', 'MA': 'Massachusetts', 'MD': 'Maryland',
    'ME': 'Maine', 'MI': 'Michigan', 'MN': 'Minnesota',
    'MO': 'Missouri', 'MS': 'Mississippi', 'MT': 'Montana',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'NE': 'Nebraska',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico',
    'NV': 'Nevada', 'NY': 'New York', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island',
    'SC': 'South Carolina', 'SD': 'South Dakota', 'TN': 'Tennessee',
    'TX': 'Texas', 'UT': 'Utah', 'VA': 'Virginia', 'VT': 'Vermont',
    'WA': 'Washington', 'WI': 'Wisconsin', 'WV': 'West Virginia',
    'WY': 'Wyoming'
}
df["state"] = df["state"].map(state_map)
# 'sex' sütunundaki değerleri yeni değerlerle eşleyelim
df.head()

,state,gender,year,name,count
0,Alaska,F,1910,Mary,14
1,Alaska,F,1910,Annie,12
2,Alaska,F,1910,Anna,10
3,Alaska,F,1910,Margaret,8
4,Alaska,F,1910,Helen,7


In [4]:
# Rename gender labels
df['gender'] = df['gender'].map({'F': 'female', 'M': 'male'})
# Generate gender based total counts
df['total_count'] = df.groupby(['state','gender', 'year'])['count'].transform('sum')
df.head()

,state,gender,year,name,count,total_count
0,Alaska,female,1910,Mary,14,68
1,Alaska,female,1910,Annie,12,68
2,Alaska,female,1910,Anna,10,68
3,Alaska,female,1910,Margaret,8,68
4,Alaska,female,1910,Helen,7,68


In [13]:
df[df["state"]=="Pennsylvania"].shape

(216171, 7)

In [5]:
# Create rank column
df['rank'] = df.groupby(['year', 'gender','state'])['count'].rank(ascending=False, method='min').astype(int)

# Sonuçları 'count' ve 'rank' sütunlarına göre sıralayarak kontrol edelim
df = df.sort_values(by=['year', 'gender', 'rank'])

In [17]:
df[(df["year"]==2025) & (df["gender"]=="female")].head(44)

,state,gender,year,name,count,total_count,rank


In [18]:
# Save to Parquet
df.to_parquet("data/preprocessed/usa/names_usa_states.parquet", engine="pyarrow", index=False)

In [10]:
# Read the saved file for checking
# import polars as pl
df = pl.read_parquet("data/preprocessed/usa/names_usa_states.parquet")
df.head()

state,gender,year,name,count,total_count,rank
str,str,i64,str,i64,i64,i32
"""Alaska""","""female""",1910,"""Mary""",14,68,1
"""Alabama""","""female""",1910,"""Mary""",875,13102,1
"""Arkansas""","""female""",1910,"""Mary""",408,7539,1
"""Arizona""","""female""",1910,"""Mary""",74,586,1
"""California""","""female""",1910,"""Mary""",295,5950,1


In [16]:
temp = df.filter( pl.col("year") ==1915)
temp.filter(pl.col("name")=="Ethan")

state,gender,year,name,count,total_count,rank
str,str,i64,str,i64,i64,i32


In [21]:
import geopandas as gpd

url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
gdf = gpd.read_file(url)[["name","geometry"]]
gdf.head()

,name,geometry
0,Alabama,"POLYGON ((-87.3593 35.00118, -85.60668 34.9847..."
1,Alaska,"MULTIPOLYGON (((-131.60202 55.11798, -131.5691..."
2,Arizona,"POLYGON ((-109.0425 37.00026, -109.04798 31.33..."
3,Arkansas,"POLYGON ((-94.47384 36.50186, -90.15254 36.496..."
4,California,"POLYGON ((-123.23326 42.00619, -122.37885 42.0..."


In [22]:
import geopandas as gpd
gdf.rename(columns={"name":"state"},inplace=True)
gdf.to_parquet("data/preprocessed/usa/usa_states_geo.parquet")

print(gdf.head())

        state                                           geometry
0     Alabama  POLYGON ((-87.3593 35.00118, -85.60668 34.9847...
1      Alaska  MULTIPOLYGON (((-131.60202 55.11798, -131.5691...
2     Arizona  POLYGON ((-109.0425 37.00026, -109.04798 31.33...
3    Arkansas  POLYGON ((-94.47384 36.50186, -90.15254 36.496...
4  California  POLYGON ((-123.23326 42.00619, -122.37885 42.0...


In [23]:
print(gdf.shape)

(52, 2)


# MERGING NATIONWIDE DATA

In [7]:
import pandas as pd
import os
import polars as pl
# folder containing your files
folder_path = "data/raw/names-usa/national-data"

# list all files (adjust extension if needed)
files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]

df_list = []

for file in files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path,header=None,names=["name", "gender", "count"])
    df["year"] = int(file[3:7])
    df_list.append(df)

# combine all files into one dataframe
df = pd.concat(df_list, ignore_index=True)
df.head(3)

,name,gender,count,year
0,Mary,F,7065,1880
1,Anna,F,2604,1880
2,Emma,F,2003,1880


In [8]:
# Rename gender labels
df['gender'] = df['gender'].map({'F': 'female', 'M': 'male'})
df.head(3)

,name,gender,count,year
0,Mary,female,7065,1880
1,Anna,female,2604,1880
2,Emma,female,2003,1880


In [9]:
df[df["name"]=="Mason"]

,name,gender,count,year
1312,Mason,male,22,1880
3434,Mason,male,13,1881
5528,Mason,male,12,1882
7661,Mason,male,12,1883
9833,Mason,male,14,1884
...,...,...,...,...
2071471,Mason,male,8038,2022
2088456,Mason,female,65,2023
2103396,Mason,male,7257,2023
2120717,Mason,female,51,2024


In [6]:
# Save to Parquet
df.to_parquet("data/preprocessed/usa/names_usa_nation.parquet", engine="pyarrow", index=False)